In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np

data_dir = Path("dataset")
with (data_dir / "vocabulary.json").open() as file:
    words = json.load(file)

embeddings = np.load(data_dir / "public_embeddings.npy", mmap_mode="r")
if embeddings.ndim != 2 or embeddings.shape[0] != len(words):
    raise ValueError("Invalid public embeddings")
norms = np.sqrt(np.einsum("ij,ij->i", embeddings, embeddings))
if np.any(norms == 0):
    raise ValueError("Invalid public embeddings")
similarities = np.empty((len(words), len(words)), dtype=np.float32)
for start in range(0, len(words), 64):
    stop = min(start + 64, len(words))
    np.matmul(embeddings[start:stop], embeddings.T, out=similarities[start:stop])
    similarities[start:stop] /= norms[start:stop, None]
    similarities[start:stop] /= norms[None, :]
del embeddings, norms
word_to_index = {word.casefold(): index for index, word in enumerate(words)}


class PublicEmbeddingPlayer:
    temperature = 0.005
    error_cap = 0.20
    margin_scale = 0.08
    hit_start = 0.40
    hit_slope = 0.40
    hit_cap = 4.80
    chunk_size = 128

    def __init__(self):
        self.count = len(words)
        self.weights = np.full(
            self.count, 1.0 / self.count, dtype=np.float64
        )
        self.used = np.zeros(self.count, dtype=bool)
        self.last_proposal = -1

    def _remove_last_proposal(self):
        if self.last_proposal >= 0:
            self.used[self.last_proposal] = True
            self.weights[self.last_proposal] = 0.0

    def _observe(self, message):
        self._remove_last_proposal()
        if message.get("verdict") == "same":
            total = float(self.weights.sum())
            if total > 0.0:
                self.weights /= total
            return
        winner = word_to_index[message["winner_word"].casefold()]
        first = word_to_index[message["word1"].casefold()]
        second = word_to_index[message["word2"].casefold()]
        if winner == first:
            loser = second
        elif winner == second:
            loser = first
        else:
            return
        margin = similarities[:, winner] - similarities[:, loser]
        scaled = np.clip(margin / self.temperature, -60.0, 60.0)
        likelihood = 1.0 / (1.0 + np.exp(-scaled))
        reliability = np.clip(
            np.abs(margin) / self.margin_scale, 0.0, 1.0
        )
        error = self.error_cap * (1.0 - reliability)
        likelihood = 0.5 * error + (1.0 - error) * likelihood
        self.weights *= likelihood
        self.weights[self.used] = 0.0
        total = float(self.weights.sum())
        if not np.isfinite(total) or total <= 1e-300:
            available = ~self.used
            self.weights.fill(0.0)
            self.weights[available] = 1.0 / int(available.sum())
        else:
            self.weights /= total

    def _choose(self, champion, turn):
        posterior = self.weights
        available = np.flatnonzero(~self.used)
        if available.size == 0:
            available = np.arange(self.count)
        if turn >= 30:
            order = np.lexsort((available, -posterior[available]))
            return int(available[order[0]])
        split = np.empty(available.size, dtype=np.float64)
        champion_scores = similarities[:, champion]
        for start in range(0, available.size, self.chunk_size):
            stop = min(start + self.chunk_size, available.size)
            candidates = available[start:stop]
            partitions = champion_scores[:, None] >= similarities[:, candidates]
            split[start:stop] = posterior @ partitions
        split = np.clip(split, 1e-12, 1.0 - 1e-12)
        entropy = -(
            split * np.log(split)
            + (1.0 - split) * np.log(1.0 - split)
        )
        hit_bonus = min(
            self.hit_cap, self.hit_start + self.hit_slope * turn
        )
        utility = entropy + hit_bonus * posterior[available]
        order = np.lexsort((available, -posterior[available], -utility))
        return int(available[order[0]])

    def respond(self, message):
        self._observe(message)
        champion = word_to_index[message["winner_word"].casefold()]
        proposal = self._choose(champion, int(message.get("turn", 1)))
        self.last_proposal = proposal
        return words[proposal]


def run_interactive():
    player = PublicEmbeddingPlayer()
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            message = json.loads(line)
        except json.JSONDecodeError:
            continue
        event = message.get("event")
        if event == "done":
            break
        if event == "new_game":
            player = PublicEmbeddingPlayer()
            continue
        if "status" in message:
            continue
        if int(message.get("turn", 0)) == 1:
            player = PublicEmbeddingPlayer()
        print(json.dumps({"new_word": player.respond(message)}), flush=True)


if "__file__" in globals():
    run_interactive()
